In [ ]:
import scanpy as sc
import pandas as pd
import plotnine as gg
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

In [ ]:
adata_pred = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/results/gaussian_ds_rollout_full/adata_pred.h5ad"
)

adata_pred_nb = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/results/nb_ds_norollout_full_rollout_eval/adata_pred.h5ad"
)

adata_test = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/data/adata_tf_test_0.h5ad"
)
adata_control = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/data/adata_control.h5ad"
)
adata_train = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/data/adata_train_0.h5ad"
)
adata_tf_train = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/data/adata_tf_train_0.h5ad"
)

In [ ]:
# amat = np.load("/workspace/experiments/06052026_cellbox_noise/data/amask.npy")
# amat_df = pd.DataFrame(
#     amat, columns=adata_pred_nb.var_names, index=adata_pred_nb.var_names
# )

# from cellbox import CellBoxEstimator

# model = CellBoxEstimator.load(
#     "/workspace/experiments/06052026_cellbox_noise/results/gaussian_ds_rollout_full/checkpoint"
# )

# params = model.state.params
# gene_names = model.adata.var_names

# # interaction matrix (masked, diagonal zeroed) — same as model.get_Amat()
# amat_df = model.get_Amat()  # pd.DataFrame, shape (G, G)

# # per-gene params as Series
# bias_s = pd.Series(np.asarray(params["b_"]), index=gene_names, name="bias")
# epsilon_s = pd.Series(
#     np.exp(np.asarray(params["epsilon_"])), index=gene_names, name="epsilon"
# )  # exp → expression scale
# overdisp_s = pd.Series(
#     np.exp(np.asarray(params["overdispersion_"])),
#     index=gene_names,
#     name="overdispersion",
# )

# # frozen stats (not trained, but useful for reference)
# x_mean_s = pd.Series(np.asarray(params["x_mean_"]), index=gene_names, name="x_mean")
# x_std_s = pd.Series(np.asarray(params["x_std_"]), index=gene_names, name="x_std")

# # look up a specific (regulator → target) edge
# gene_reg = "laci"
# gene_targ = "laca"
# reg_id = amat_df.index.get_loc(gene_reg)
# targ_id = amat_df.columns.get_loc(gene_targ)

# edge_weight = amat_df.iloc[reg_id, targ_id]  # scalar

# gene = "lacz"

# adata_pred[adata_pred.obs["target"] == "laci", gene].X.mean(axis=0), adata_test[
#     adata_test.obs["target"] == "laci", gene
# ].layers["log1p"].mean(axis=0)

In [ ]:
for target in adata_test.obs["target"].unique():
    print(target)

In [ ]:
adata_pred

In [ ]:
def compute_lfcs(adata_pred):
    res = []
    for target in adata_test.obs["target"].unique():
        lfc_pred = adata_pred[adata_pred.obs["target"] == target].X.mean(
            axis=0
        ) - adata_control.layers["log1p"].mean(axis=0)
        lfc_true = adata_test[adata_test.obs["target"] == target].layers["log1p"].mean(
            axis=0
        ) - adata_control.layers["log1p"].mean(axis=0)

        df = pd.DataFrame(
            {
                "lfc_pred": lfc_pred,
                "lfc_true": lfc_true,
                "gene": adata_test.var_names,
            }
        ).assign(target=target)
        res.append(df)
    res = pd.concat(res)
    return res

In [ ]:
res_gaussian = compute_lfcs(adata_pred).assign(model="gaussian")
res_nb = compute_lfcs(adata_pred_nb).assign(model="nb")
res = pd.concat([res_gaussian, res_nb]).dropna()

# res = pd.concat([res_nb]).dropna()
res

In [ ]:
from cellbox import CellBoxEstimator

model = CellBoxEstimator.load(
    "/workspace/experiments/06052026_cellbox_noise/results/nb_ds_rollout_full_v2/checkpoint"
)

In [ ]:
overd = np.array(model.state.params["overdispersion_"])
overd = np.exp(overd)
overd_df = pd.DataFrame({"gene": model.adata.var_names, "overdispersion": overd})

In [ ]:
res_ = res.merge(overd_df, on="gene", how="left")
res_nb = res_[res_["model"] == "nb"]
plt.scatter(res_nb["overdispersion"], np.abs(res_nb["lfc_true"] - res_nb["lfc_pred"]))
plt.show()

In [ ]:
res.groupby(["model", "target"]).apply(
    lambda df: stats.pearsonr(df["lfc_true"], df["lfc_pred"])[0]
).to_frame("pearson_r").reset_index().sort_values("pearson_r", ascending=False)

In [ ]:
pd.set_option("display.max_rows", 100)
adata_test.obs.target.value_counts().head(100)

In [ ]:
# plot_df = res_nb.query("target == 'aaer'").copy()
# plot_df = res_gaussian.query("target == 'hicb'").copy()
plot_df = res_gaussian.query("target == 'laci'").copy()

(
    gg.ggplot(plot_df, gg.aes(x="lfc_true", y="lfc_pred"))
    + gg.geom_point()
    + gg.theme_bw()
)

In [ ]:
import plotly.express as px

fig = px.scatter(plot_df, x="lfc_true", y="lfc_pred", hover_data=["gene"])
fig.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

# kmeans = KMeans(n_clusters=3, random_state=0).fit(plot_df[["lfc_true", "lfc_pred"]])
# plot_df["cluster"] = kmeans.labels_

kmeans = GaussianMixture(n_components=4, covariance_type="full").fit(
    plot_df[["lfc_true", "lfc_pred"]], plot_df["gene"]
)
plot_df["cluster"] = kmeans.predict(plot_df[["lfc_true", "lfc_pred"]])

(
    gg.ggplot(plot_df, gg.aes(x="lfc_true", y="lfc_pred", color="factor(cluster)"))
    + gg.geom_point()
    + gg.theme_bw()
)

In [ ]:
plot_df.sort_values("lfc_true", ascending=False).head(20)

In [ ]:
plot_df.sort_values("lfc_true", ascending=True).head(20)

In [ ]:
", ".join(plot_df.sort_values("lfc_true", ascending=True)["gene"].head(20))

In [ ]:
amat_df.loc["laca"].loc["laci"]

In [ ]:
amat_df.loc["lacy"].loc["laci"]

In [ ]:
amat_df.loc["lacz"].loc["laci"]

In [ ]:
reg_expression_train = adata_train[:, "laci"].layers["log1p"].toarray().squeeze()
reg_expression_control = adata_control[:, "laci"].layers["log1p"].toarray().squeeze()
# reg_expression_test = adata_test[adata_test.target, "laci"].layers["log1p"].toarray().squeeze()

bins = np.linspace(0, 3.5, 50)
plt.hist(
    reg_expression_train,
    bins=bins,
    density=True,
    alpha=0.5,
    label="train data (excludes lacI KD)",
)
plt.hist(
    reg_expression_control,
    bins=bins,
    density=True,
    alpha=0.5,
    label="control data (no perturbation)",
)
plt.xlabel("lacI expression (log-CP10K)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig("laci_expression_histogram.svg")
plt.show()

In [ ]:
print("lacI expression in training data (no lacI KD):")
display(pd.Series(reg_expression_train).describe())

In [ ]:
lib_train = adata_train.layers["reads"].sum(axis=1).A1
reg_expression_train = adata_train[:, "laca"].layers["log1p"].toarray().squeeze()

plt.scatter(lib_train, reg_expression_train)
plt.xscale("log")
plt.title("laca expression vs library size in training data")
plt.xlabel("Library size (log scale)")
plt.ylabel("laca expression (log-CP10K)")
plt.show()

lib_control = adata_control.layers["reads"].sum(axis=1).A1
reg_expression_control = adata_control[:, "laca"].layers["log1p"].toarray().squeeze()
plt.scatter(lib_control, reg_expression_control)
plt.xscale("log")
plt.title("laca expression vs library size in control data")
plt.xlabel("Library size (log scale)")
plt.ylabel("laca expression (log-CP10K)")
plt.show()


lib_test = adata_test.layers["reads"].sum(axis=1).A1
reg_expression_test = adata_test[:, "laca"].layers["log1p"].toarray().squeeze()
plt.scatter(lib_test, reg_expression_test)
plt.xscale("log")
plt.title("laca expression vs library size in test data")
plt.xlabel("Library size (log scale)")
plt.ylabel("laca expression (log-CP10K)")
plt.show()

In [ ]:
reg_expression_train = adata_train[:, "laci"].layers["log1p"].toarray().squeeze()
targ_expression_train = adata_train[:, "laca"].layers["log1p"].toarray().squeeze()

reg_expression_control = adata_control[:, "laci"].layers["log1p"].toarray().squeeze()
targ_expression_control = adata_control[:, "laca"].layers["log1p"].toarray().squeeze()

reg_expression_test = adata_test[:, "laci"].layers["log1p"].toarray().squeeze()
targ_expression_test = adata_test[:, "laca"].layers["log1p"].toarray().squeeze()

print("Correlation between lacI and lacA expression in training data (no lacI KD):")
print(stats.pearsonr(reg_expression_train, targ_expression_train))

print("Correlation between lacI and lacA expression in control data (no perturbation):")
print(stats.pearsonr(reg_expression_control, targ_expression_control))
print("Correlation between lacI and lacA expression in test data (includes lacI KD):")
print(stats.pearsonr(reg_expression_test, targ_expression_test))

plt.scatter(
    reg_expression_train,
    targ_expression_train,
    color="blue",
    alpha=0.5,
    label="training data (no lacI KD)",
)
plt.scatter(
    reg_expression_control,
    targ_expression_control,
    color="orange",
    alpha=0.5,
    label="control data (no perturbation)",
)
plt.scatter(
    reg_expression_test,
    targ_expression_test,
    color="green",
    alpha=0.5,
    label="test data (includes lacI KD)",
)

plt.legend()
plt.xlabel("lacI expression (log-CP10K)")
plt.ylabel("lacA expression (log-CP10K)")
plt.title("lacA vs lacI expression across datasets")
plt.tight_layout()
plt.show()

In [ ]:
layer = "log1p"
x1 = adata_test[:, "laci"].layers[layer].toarray().squeeze()
x2 = adata_test[:, "lacz"].layers[layer].toarray().squeeze()
plt.scatter(x1, x2)
import scipy.stats as stats

print("Pearson correlation:", stats.pearsonr(x1, x2))
plt.show()

In [ ]:
layer = "log1p"
x1 = adata_train[:, "laci"].layers[layer].toarray().squeeze()
x2 = adata_train[:, "laca"].layers[layer].toarray().squeeze()
plt.scatter(x1, x2, label="train data (excludes lacI KD)")
plt.xlabel("lacI expression (log-CP10K)")
plt.ylabel("lacA expression (log-CP10K)")
import scipy.stats as stats

plt.legend()

print("Pearson correlation:", stats.pearsonr(x1, x2))
plt.show()